In [1]:
import logging
import os
from contextlib import contextmanager
import sys
import pandas as pd
import numpy as np

logging.getLogger().setLevel(logging.ERROR)

In [2]:
@contextmanager
def suppress_print():
    original_stdout = sys.stdout
    sys.stdout = open(os.devnull, 'w')
    try:
        yield
    finally:
        sys.stdout.close()
        sys.stdout = original_stdout
        
with suppress_print():
    from fr3d.pdb.pdb_reader import PDBStructure
    print("This will be suppressed")

In [3]:
pdb_data_dir = 'path/to/pdbs/'   # the path for your pdbs
csv_data_path = 'path/to/csvs/'  # the path for your pdbs corresponding csv with heads like "pdb_id, sequence"
sequence_cluster_file_path = 'path/to/sequence_clusters/' # the path for your pdbs corresponding sequence cluster with tools as paper states
structure_cluster_file_path = 'path/to/structure_clusters/' # the path for your pdbs corresponding structure cluster with tools as paper states


In [ ]:
data = pd.read_csv(csv_data_path)

with open(sequence_cluster_file_path, 'r') as file:
    lines = file.readlines()

clusters = []

# process each pdb for sequence cluster center
for i, line in enumerate(lines):
    pdbs = line.strip().split('\t')
    if pdbs:  
        # use first PDB file as cluster center
        cluster_center = pdbs[0].strip()
        for pdb in pdbs[1:]: 
            clusters.append({
                'Center': cluster_center,
                'pdb_id': pdb.strip()
            })
seq_cluster_df = pd.DataFrame(clusters)


with open(structure_cluster_file_path, 'r') as file:
    lines = file.readlines()
clusters = []

# process each pdb for structure cluster center
for i, line in enumerate(lines):
    pdbs = line.strip().split('\t')
    if pdbs:
        cluster_center = pdbs[0][1:].strip()[:-4]
        for pdb in pdbs:
            clusters.append({
                'Center': cluster_center,
                'pdb_id': pdb[1:].strip()[:-4]
            })

struture_cluster_df = pd.DataFrame(clusters)

In [ ]:
seq_cluster_dict = {}
cnt = 0
for cluster in seq_cluster_df['Center'].to_list():
    if cluster not in seq_cluster_dict.keys():
        seq_cluster_dict[cluster] = cnt
        cnt += 1
seq_cluster_df['seq_cluster'] = seq_cluster_df['Center'].apply(lambda x: seq_cluster_dict[x])
seq_cluster_df = seq_cluster_df[['pdb_id', 'seq_cluster']]

structure_cluster_dict = {}
cnt = 0
for cluster in struture_cluster_df['Center'].to_list():
    if cluster not in structure_cluster_dict.keys():
        structure_cluster_dict[cluster] = cnt
        cnt += 1
struture_cluster_df['structure_cluster'] = struture_cluster_df['Center'].apply(lambda x: structure_cluster_dict[x])
struture_cluster_df = struture_cluster_df[['pdb_id', 'structure_cluster']]

In [ ]:
new_data = pd.merge(data, struture_cluster_df, on='pdb_id', how='inner')
new_data = new_data.dropna()
new_data = new_data.drop_duplicates(subset='pdb_id', keep='first')

final_data = pd.merge(new_data, seq_cluster_df, on='pdb_id', how='inner')
final_data = final_data.dropna()
final_data = final_data.drop_duplicates(subset='pdb_id', keep='first')

In [ ]:
train_ratio, valid_ratio, test_ratio = 0.7, 0.2, 0.1
all_structure_cluster = list(set(final_data['structure_cluster'].to_list()))

#first we use structure cluster label to allocate train/valid/test label
train_size = int(len(all_structure_cluster) * train_ratio)
valid_size = int(len(all_structure_cluster) * valid_ratio)
test_size = len(all_structure_cluster) - train_size - valid_size

train_structure_cluster = np.random.choice(all_structure_cluster, size=train_size, replace=False)
remaining_indices = np.setdiff1d(all_structure_cluster, train_structure_cluster)
valid_structure_cluster = np.random.choice(remaining_indices, size=valid_size, replace=False)
test_structure_cluster = np.setdiff1d(remaining_indices, valid_structure_cluster)

In [ ]:
def structure_label(cluster):
    if cluster in train_structure_cluster:
        return 'train'
    elif cluster in valid_structure_cluster:
        return 'valid'
    elif cluster in test_structure_cluster:
        return 'test'
    raise ValueError("not valid indices")

In [ ]:
final_data['split'] = final_data['structure_cluster'].apply(lambda x: structure_label(x))
train_df = final_data[final_data['split']=='train'].reset_index(drop=True)
train_seq_cluster = list(set(train_df['seq_cluster'].to_list()))

# delete valid dataset pdb where its sequence cluster appear in train dataset
valid_df = final_data[final_data['split']=='valid'].reset_index(drop=True)
filter_valid_df = valid_df[~valid_df['seq_cluster'].isin(train_seq_cluster)]

# delete test dataset pdb where its sequence cluster appear in train/valid dataset
valid_seq_cluster = list(set(filter_valid_df['seq_cluster'].to_list()))
train_valid_seq_cluster = train_seq_cluster + valid_seq_cluster
test_df = final_data[final_data['split']=='test'].reset_index(drop=True)
filter_test_df = test_df[~test_df['seq_cluster'].isin(train_valid_seq_cluster)]

In [ ]:
train_df = train_df[['pdb_id', 'sequence']].reset_index(drop=True)
valid_df = filter_valid_df[['pdb_id', 'sequence']].reset_index(drop=True)
test_df = filter_test_df[['pdb_id', 'sequence']].reset_index(drop=True)

train_df.to_csv('path/to/train.csv', index=False)
valid_df.to_csv('path/to/valid.csv', index=False)
test_df.to_csv('path/to/test.csv', index=False)